# 02 - Register and verify the external agent

The registration is metadata. It links the stable `gen_ai.agent.id` emitted by the external runtime to the Foundry agent experience.

In [ ]:
import os
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ExternalAgentDefinition
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

if not Path('.env').exists():
    raise FileNotFoundError('Copy .env.example to .env and fill in the required values.')

load_dotenv('.env')

required_variables = ('FOUNDRY_PROJECT_ENDPOINT', 'AGENT_NAME', 'OTEL_AGENT_ID')
missing = [name for name in required_variables if not os.getenv(name)]
if missing:
    raise ValueError(f'Missing required environment variables: {missing}')

project = AIProjectClient(
    endpoint=os.environ['FOUNDRY_PROJECT_ENDPOINT'],
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

In [ ]:
agent_name = os.environ['AGENT_NAME']
otel_agent_id = os.environ['OTEL_AGENT_ID']

existing_names = {agent.name for agent in project.agents.list(kind='external')}

if agent_name not in existing_names:
    registered = project.agents.create_version(
        agent_name=agent_name,
        description='External agent registered for Foundry observability and evaluation.',
        definition=ExternalAgentDefinition(otel_agent_id=otel_agent_id),
    )
    print(f'Registered external agent: {registered.name}')
else:
    print(f'External agent already registered: {agent_name}')

In [ ]:
registered = project.agents.get(agent_name=agent_name)
registered_otel_id = registered.versions.latest.definition.otel_agent_id

if registered_otel_id != otel_agent_id:
    raise ValueError(
        'Registration mismatch: '
        f'Foundry has {registered_otel_id!r}, but .env has {otel_agent_id!r}.'
    )

print(f'Agent name: {registered.name}')
print(f'Registered OpenTelemetry ID: {registered_otel_id}')
print('Registration matches the local telemetry configuration.')

## Verify traces

After telemetry has had 2-5 minutes to ingest:

1. Open [Microsoft Foundry](https://ai.azure.com).
2. Open this project.
3. Select **Build → Agents** and open the registered agent.
4. Select **Traces**.

If no trace appears, confirm that the project is connected to the same Application Insights resource used in `.env`, and that the emitted `gen_ai.agent.id` exactly matches the registered OpenTelemetry ID.

In [ ]:
# Optional cleanup. Uncomment only when you intend to remove the registration.
# This does not stop or delete the externally hosted runtime.
# project.agents.delete(agent_name=agent_name, force=True)